In [1]:
from pathlib import Path
import sys


from deepeval.synthesizer import Synthesizer
from sqlalchemy import text
from sqlalchemy import create_engine
from sqlalchemy.orm import Session

import os
import json
from deepeval.models import GPTModel
from deepeval.synthesizer import Synthesizer
from deepeval.models import DeepEvalBaseLLM

In [2]:
DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL, echo=False)


In [3]:
def fetch_all_entities_with_chunks():
    """
    Fetch all pharaohs and landmarks with their text chunks
    Filters out empty/meaningless chunks (less than 5 characters)
    """
    print("Fetching all entities from database...")
    
    entities_data = []
    
    with Session(engine) as session:
        # Fetch all pharaohs with their chunks
        pharaohs_query = """
        SELECT 
            p.name,
            pt.text_chunk
        FROM pharaohs p
        JOIN pharaohs_texts pt ON p.id = pt.pharaoh_id
        ORDER BY p.name
        """
        
        pharaoh_results = session.execute(text(pharaohs_query)).fetchall()
        
        # Group chunks by pharaoh and filter
        pharaoh_chunks = {}
        for name, chunk in pharaoh_results:
            if chunk and len(chunk.strip()) >= 5:
                if name not in pharaoh_chunks:
                    pharaoh_chunks[name] = []
                pharaoh_chunks[name].append(chunk.strip())
        
        # Only keep pharaohs with at least 1 valid chunk
        for name, chunks in pharaoh_chunks.items():
            if chunks:
                entities_data.append({
                    "entity_type": "pharaoh",
                    "entity_name": name,
                    "contexts": chunks
                })
        
        print(f"  ✓ Fetched {len(pharaoh_chunks)} pharaohs with valid chunks")
        
        # Fetch all landmarks with their chunks
        landmarks_query = """
        SELECT 
            l.name,
            lt.text_chunk
        FROM landmarks l
        JOIN landmarks_texts lt ON l.id = lt.landmark_id
        ORDER BY l.name
        """
        
        landmark_results = session.execute(text(landmarks_query)).fetchall()
        
        # Group chunks by landmark and filter
        landmark_chunks = {}
        for name, chunk in landmark_results:
            # Skip if chunk is None or too short
            if chunk and len(chunk.strip()) >= 5:
                if name not in landmark_chunks:
                    landmark_chunks[name] = []
                landmark_chunks[name].append(chunk.strip())
        
        # Only keep landmarks with at least 1 valid chunk
        for name, chunks in landmark_chunks.items():
            if chunks:
                entities_data.append({
                    "entity_type": "landmark",
                    "entity_name": name,
                    "contexts": chunks
                })
        
        print(f"  ✓ Fetched {len(landmark_chunks)} landmarks with valid chunks")
    
    print(f"\nTotal entities with valid chunks: {len(entities_data)}")
    return entities_data

In [4]:
entities_data=fetch_all_entities_with_chunks()

Fetching all entities from database...
  ✓ Fetched 80 pharaohs with valid chunks
  ✓ Fetched 52 landmarks with valid chunks

Total entities with valid chunks: 132


In [32]:
class GroqModel(DeepEvalBaseLLM):
    def __init__(self, model_name="moonshotai/kimi-k2-instruct-0905"):
        from groq import Groq
        self.client = Groq(api_key=os.getenv("GROQ_API_KEY3"))
        self.model_name = model_name
    
    def load_model(self):
        return self.model_name
    
    def generate(self, prompt: str) -> str:
        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7
        )
        return response.choices[0].message.content
    
    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)
    
    def get_model_name(self):
        return self.model_name

# Use it with synthesizer
groq_model = GroqModel()
synthesizer = Synthesizer(model=groq_model)

In [39]:
"""def generate_synthetic_dataset(entities_data, questions_per_entity=10):
    
    Generate synthetic questions for each entity using DeepEval Synthesizer with Groq
    
    print(f"\nGenerating questions per entity...")
    
    # Initialize Groq model
    groq_model = GroqModel()
    synthesizer = Synthesizer(model=groq_model)
    
    all_goldens = []
    
    for i, entity in enumerate(entities_data, 1):
        print(f"[{i}/{len(entities_data)}] Generating questions for {entity['entity_name']}...")
        
        entity_goldens = []
        
        # Generate from each context individually
        for idx, context in enumerate(entity['contexts']):
            try:
                # Pass single context as a list with one element
                goldens = synthesizer.generate_goldens_from_contexts(
                    contexts=[context],  # Wrap single context in a list
                    include_expected_output=True
                )
                
                entity_goldens.extend(goldens)
                
                # Stop if we have enough questions
                if len(entity_goldens) >= questions_per_entity:
                    break
                    
            except Exception as e:
                print(f"  ✗ Error on context {idx}: {str(e)[:80]}")
                continue
        
        # Limit to questions_per_entity
        entity_goldens = entity_goldens[:questions_per_entity]
        
        # Add entity metadata to each golden
        for golden in entity_goldens:
            if not hasattr(golden, 'additional_metadata'):
                golden.additional_metadata = {}
            golden.additional_metadata.update({
                "entity_type": entity['entity_type'],
                "entity_name": entity['entity_name']
            })
        
        all_goldens.extend(entity_goldens)
        print(f"  ✓ Generated {len(entity_goldens)} questions")
    
    print(f"\nTotal questions generated: {len(all_goldens)}")
    return all_goldens"""

'def generate_synthetic_dataset(entities_data, questions_per_entity=10):\n\n    Generate synthetic questions for each entity using DeepEval Synthesizer with Groq\n\n    print(f"\nGenerating questions per entity...")\n\n    # Initialize Groq model\n    groq_model = GroqModel()\n    synthesizer = Synthesizer(model=groq_model)\n\n    all_goldens = []\n\n    for i, entity in enumerate(entities_data, 1):\n        print(f"[{i}/{len(entities_data)}] Generating questions for {entity[\'entity_name\']}...")\n\n        entity_goldens = []\n\n        # Generate from each context individually\n        for idx, context in enumerate(entity[\'contexts\']):\n            try:\n                # Pass single context as a list with one element\n                goldens = synthesizer.generate_goldens_from_contexts(\n                    contexts=[context],  # Wrap single context in a list\n                    include_expected_output=True\n                )\n\n                entity_goldens.extend(goldens)\n\n

In [40]:
import json
import time
from groq import Groq

def generate_synthetic_dataset(entities_data, questions_per_entity=10):
    """
    Generate synthetic questions for each entity using Groq directly
    """
    print(f"\nGenerating {questions_per_entity} questions per entity using Groq...")
    
    client = Groq(api_key=os.getenv("GROQ_API_KEY3"))
    all_test_cases = []
    
    for i, entity in enumerate(entities_data, 1):
        print(f"[{i}/{len(entities_data)}] Generating for {entity['entity_name']}...")
        
        # Combine all contexts for this entity
        combined_context = "\n\n---\n\n".join(entity['contexts'])
        
        prompt = f"""You are generating evaluation questions for a RAG system about Ancient Egypt.

Given the following information about {entity['entity_name']}:

{combined_context}

Generate {questions_per_entity} diverse questions that can be answered using ONLY the information provided above.
For each question, provide the expected answer based strictly on the context.

Requirements:
- Mix of factual, reasoning, and comparison questions
- Questions should be specific to {entity['entity_name']}
- Answers must be derived from the provided context only

Format your response as a valid JSON array:
[
  {{"question": "...", "answer": "..."}},
  {{"question": "...", "answer": "..."}}
]

Return ONLY the JSON array, nothing else."""

        try:
            response = client.chat.completions.create(
                model="moonshotai/kimi-k2-instruct-0905",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.8,
                max_tokens=2000
            )
            
            # Parse JSON response
            response_text = response.choices[0].message.content.strip()
            
            # Extract JSON if it's wrapped in markdown
            if response_text.startswith("```json"):
                response_text = response_text.split("```json")[1].split("```")[0].strip()
            elif response_text.startswith("```"):
                response_text = response_text.split("```")[1].split("```")[0].strip()
            
            qa_pairs = json.loads(response_text)
            
            # Add to dataset
            for qa in qa_pairs:
                all_test_cases.append({
                    "input": qa["question"],
                    "expected_output": qa["answer"],
                    "context": entity['contexts'],
                    "entity_type": entity['entity_type'],
                    "entity_name": entity['entity_name']
                })
            
            print(f"  ✓ Generated {len(qa_pairs)} questions")
            
            # Add delay to avoid rate limits
            time.sleep(2)
            
        except json.JSONDecodeError as e:
            print(f"  ✗ JSON parsing error: {e}")
            print(f"  Response: {response_text[:200]}...")
        except Exception as e:
            print(f"  ✗ Error: {e}")
            continue
    
    print(f"\nTotal questions generated: {len(all_test_cases)}")
    return all_test_cases

In [ ]:
from deepeval.dataset import EvaluationDataset, Golden
from pathlib import Path

def save_and_create_deepeval_dataset(goldens_list, output_dir="evaluation_data"):
    """
    Converts raw list of dicts to a DeepEval EvaluationDataset and saves it.
    """
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True)
    
    # 1. Convert dicts into DeepEval 'Golden' objects
    deepeval_goldens = []
    for item in goldens_list:
        golden = Golden(
            input=item["input"],
            expected_output=item["expected_output"],
            context=item["context"], # The list of chunks used to generate it
            additional_metadata={
                "entity_name": item["entity_name"],
                "entity_type": item["entity_type"]
            }
        )
        deepeval_goldens.append(golden)
    
    # 2. Create the Dataset object
    dataset = EvaluationDataset(goldens=deepeval_goldens)
    
    # 3. Save as the standard DeepEval format (JSON)
    save_path = output_path / "egypt_benchmark.json"
    dataset.save_as(file_type="json", directory=str(output_path))
    
    print(f"\n✓ Created DeepEval Dataset with {len(deepeval_goldens)} test cases.")
    return dataset

# Run it


In [42]:
entities_data = fetch_all_entities_with_chunks()

Fetching all entities from database...
  ✓ Fetched 80 pharaohs with valid chunks
  ✓ Fetched 52 landmarks with valid chunks

Total entities with valid chunks: 132


In [43]:
entities_data=[entities_data[0]]

In [44]:
print(entities_data)

[{'entity_type': 'pharaoh', 'entity_name': 'Achoris', 'contexts': ['Domestically, Hakoris’s eighth regnal year was marked by a significant internal rebellion that was suppressed by General Nakhtnebef. Despite continued Persian aggression, Hakoris maintained Egyptian sovereignty and ensured relative peace and prosperity within his borders. His reign coincided with a period of complex international relations, including peace between Persia and Sparta but intermittent warfare among Greek city-states. Egypt under Hakoris emerged as a significant regional power allied with Greek and Cypriot interests, maintaining stability and independence during a time of shifting alliances across the eastern Mediterranean.', 'Politically, Hakoris skillfully balanced diplomacy and military preparedness to preserve Egypt’s independence from Persia. During his reign, Egypt remained free from Persian control. He concluded an alliance with Athens and employed a large mercenary army composed of seasoned Greek v

In [45]:
goldens = generate_synthetic_dataset(entities_data, questions_per_entity=10)


Generating 10 questions per entity using Groq...
[1/1] Generating for Achoris...
  ✓ Generated 10 questions

Total questions generated: 10


In [ ]:
dataset = save_and_create_deepeval_dataset(goldens)

AttributeError: 'dict' object has no attribute 'input'